<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l5.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L5 · Testnet demo antes de stake
Pipeline demo end-to-end y umbral de paso.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l5.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l5.csv'), Path('data/c6_l5.csv'), Path('c6_l5.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))


In [ ]:
# Pipeline demo: generar -> validar -> puntuar (stake = 0)
df['prediction'] = df['score_demo'].rank(pct=True)
sub = pd.DataFrame({'id': 'demo_' + df['dia'].astype(str), 'prediction': df['prediction']})
assert sub['prediction'].between(0, 1).all() and sub['id'].is_unique and sub.notna().all().all()
df['semana'] = (df['dia'] - 1) // 7 + 1
por_sem = df.groupby('semana')['score_demo'].mean()
print(por_sem.round(4).to_string())
print('Score demo medio:', round(por_sem.mean(), 4))


In [ ]:
# Umbral de paso: media > 0.01 y 4/6 últimas semanas en positivo
ventanas = [por_sem.iloc[i:i+6] for i in range(len(por_sem) - 5)]
pasan = sum((v.mean() > 0.01) and ((v > 0).sum() >= 4) for v in ventanas)
print(f'Ventanas de 6 semanas: {len(ventanas)}, pasan el umbral: {pasan}')
assert isinstance(pasan, (int, np.integer))


In [ ]:
# Umbral laxo para comparar: media > 0 deja colar ventanas malas
pasan_laxo = sum((v.mean() > 0) for v in ventanas)
print(f'Con umbral laxo pasarían: {pasan_laxo} (más ventanas, menos filtro)')
assert pasan_laxo > pasan, 'el umbral estricto debe filtrar más'
assert pasan >= 1, 'al menos una ventana debe pasar el corte'



In [ ]:
# Chequeo automático L5
assert sub['prediction'].between(0, 1).all()
assert len(ventanas) > 0 and pasan <= len(ventanas)
print(f'OK L5: pipeline demo verificado ({pasan}/{len(ventanas)} ventanas pasan)')
